# 03 — Policy Rules: a Deterministic Baseline for Expense Auditing

This notebook builds the **rules engine** that audits every generated expense claim
against the company travel & expense policy (`policy.yaml`). It is the deterministic
baseline that the LLM agent (notebook 05) will later be measured against (notebook 06).

**What it does, end to end:**

| Section | Builds | Policy sections |
|---|---|---|
| 1 | Load policy + claims (no labels) | — |
| 2 | Typed `Claim` input and `Violation` output models | — |
| 3 | Single-claim rules: late submission, missing receipt | POL-1.2, POL-1.3 |
| 4 | Category cap rules + the generic `apply_rules` runner | POL-2.2/2.3/2.4/2.7/2.9/2.10 |
| 5 | Fare/rental class, software vendor, mileage-rate rules | POL-2.1, POL-2.5, POL-2.6, POL-2.8 |
| 6 | Review *flags*: prohibited-keyword hints, unapproved vendors | POL-3.x, POL-2.8 |
| 7 | Escalation flags: high value, near cap | POL-5.1, POL-5.2 |
| 8 | Cross-claim rules: duplicates and split purchases | POL-1.4, POL-1.5 |
| 9 | `run_rules()`: one decision row per claim | POL-4, POL-5 |
| 10 | Score rule decisions against ground-truth labels | evaluation only |

**Design principles used throughout:**

- **Labels never feed a rule.** Ground-truth labels are loaded only in section 10,
  strictly for evaluation. Section 1 even asserts the claims frame carries no label column.
- **`policy.yaml` is the single source of truth.** Thresholds, caps, vendor lists, and
  violation codes all come from the loaded policy dict — never hard-coded — and a drift
  check (cell 2b) fails if the code's `ViolationCode` literal and the config disagree.
- **Violations vs flags.** A `Violation` is a broken rule that *denies* a claim; a `Flag`
  only routes it to *review*. Decision precedence is deny > review > approve.
- **Every rule cites its policy section** via a validated `policy_ref` (e.g. `POL-2.2`),
  so each finding is auditable back to the written policy.
- **Boundary tests after every rule group** (the "b" cells) pin exact thresholds
  ($75.00 vs $75.01, 60 vs 61 days) plus *anchor* assertions against known counts in
  the generated dataset, so silent regressions fail loudly.

## 1: Imports, policy, claims

Load the policy config and the 330 generated claims. Labels are loaded here only so
later cells can reuse the loader, but they are **not** merged into the claims — the
final assert proves `in_policy` never leaks into `claims_df`, keeping rule-building
honest. `%autoreload` lets edits in the `expense_agent` package take effect without
restarting the kernel.

In [1]:
# 1. Imports, policy, claims (no labels)
%load_ext autoreload
%autoreload 2

import re
from collections.abc import Callable
from collections import defaultdict
from datetime import date, timedelta
from datetime import date as Date
from typing import Any, Literal, get_args

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    field_validator,
)

from expense_agent.data import load_claims, load_labels
from expense_agent.paths import GENERATED_DIR, POLICY_PATH
from expense_agent.policy import cap_usage, load_policy

policy = load_policy(POLICY_PATH)
claims_df = load_claims(GENERATED_DIR)
labels_df = load_labels(GENERATED_DIR)

print(f"policy v{policy['version']} claims={len(claims_df)}")
assert len(claims_df) == 330, "Expected the generated set (330 claims)"
assert "in_policy" not in claims_df.columns, "Label leak in claims_df"

policy v1.0 claims=330


## 2: Typed Input

Pydantic models form the contract for everything downstream:

- **`Claim`** — one expense line item, `frozen=True` (immutable, so rules can't mutate
  inputs) and `extra="ignore"` (tolerates surplus DataFrame columns). The
  `blank_to_none` validator runs on every field *before* type coercion, converting
  pandas `NaN`/`<NA>` and empty/whitespace strings to `None` so optional fields behave.
- **`Violation`** — one broken rule on one claim. `extra="forbid"` plus a regex-validated
  `policy_ref` (`POL-d.d`) and non-empty `detail` make malformed findings impossible
  to construct.
- **`to_claims`** validates every row and fails loudly on bad data rather than skipping it.

Cell 2b verifies: all 330 rows validate with unique IDs, the `ViolationCode` literal
matches `policy.yaml`'s `violation_codes` (single source of truth), and a miscased
policy reference (`Pol-2.2`) is rejected by validation as expected.

In [2]:
# 2. Typed input (Claim) and output (Violation)
ViolationCode = Literal[
    "over_cap",
    "prohibited_category",
    "missing_receipt",
    "late_submission",
    "duplicate",
    "split_transaction",
    "personal_extension",
    "premium_class",
]


class Claim(BaseModel):
    """One submitted expense line item. Carries no labels."""

    model_config = ConfigDict(frozen=True, extra="ignore")

    report_id: str
    claim_id: str
    employee_id: str
    department: str
    submitted_date: date
    expense_date: date
    category: str
    merchant: str
    description: str
    business_purpose: str | None = None
    amount: float = Field(gt=0)
    city: str | None = None
    nights: int | None = Field(default=None, ge=0)
    attendees: int | None = Field(default=None, ge=0)
    fare_class: str | None = None
    flight_hours: float | None = None
    rental_class: str | None = None
    miles: int | None = Field(default=None, ge=0)
    preapproved: bool | None = None
    receipt_attached: bool

    @field_validator(
        "*", mode="before"
    )  # strip whitespace from all string fields
    @classmethod
    def blank_to_none(cls, value: object) -> object:
        """Turn pandas NaN/<NA> and empty strings into None."""
        if isinstance(value, str):
            return value.strip() or None
        if value is not None and pd.isna(value):
            return None
        return value


class Violation(BaseModel):
    """One broken rule on one claim, citing the policy section."""

    model_config = ConfigDict(frozen=True, extra="forbid")

    claim_id: str
    code: ViolationCode
    policy_ref: str = Field(pattern=r"^POL-\d+\.\d+$")
    detail: str = Field(min_length=1)


def to_claims(df: pd.DataFrame) -> list[Claim]:
    """Validate every row into a Claim; fail loudly on a bad row."""
    return [Claim.model_validate(row) for row in df.to_dict("records")]


claims = to_claims(claims_df)
print(f"validated {len(claims)} claims")

validated 330 claims


In [3]:
# 2b - checks
assert len(claims) == 330
assert len({c.claim_id for c in claims}) == 330, "claim_id not unique"

# Codes in code must match codes in config (single source of truth)
assert set(get_args(ViolationCode)) == set(policy["violation_codes"]), (
    "ViolationCode drifted from policy.yaml"
)

# A bad citation must be rejected
bad = {
    "claim_id": "X",
    "code": "over_cap",
    "policy_ref": "Pol-2.2",
    "detail": "test",
}
try:
    Violation(**bad)
except ValidationError as err:
    for error in err.errors():
        print(error["loc"], "->", error["msg"])
    print(f"rejected as expected: {err.error_count()} error(s)")
else:
    raise AssertionError("bad policy_ref was accepted")

('policy_ref',) -> String should match pattern '^POL-\d+\.\d+$'
rejected as expected: 1 error(s)


## 3: Single-claim rules

Defines the shared rule signature — `Rule = Callable[[Claim, Policy], Violation | None]` —
that every single-claim rule follows: look at one claim, return one violation or nothing.

- **`check_late` (POL-1.2):** flags submissions *more than* N days after the expense
  date (N comes from the policy, currently 60 — so day 60 is on time, day 61 is late).
- **`check_receipt` (POL-1.3):** flags amounts over the receipt threshold with no
  receipt attached. Mileage is exempt: it's computed as miles × rate, so no receipt exists.

Cell 3b pins the boundaries (60 vs 61 days, $75.00 vs $75.01), confirms the mileage
exemption, and anchors against the dataset: exactly 11 late claims, matching what
notebook 01b found.

In [4]:
# 3. single-claim rules: late submission, missing receipt

Policy = dict[str, Any]
Rule = Callable[[Claim, Policy], Violation | None]


def check_late(claim: Claim, policy: Policy) -> Violation | None:
    """Flag claims submitted more than N days after the expense (POL-1.2)."""
    max_days = policy["submission"]["max_days_after_expense"]
    days = (claim.submitted_date - claim.expense_date).days
    if days > max_days:
        return Violation(
            claim_id=claim.claim_id,
            code="late_submission",
            policy_ref="POL-1.2",
            detail=f"submitted {days} days after expense (limit {max_days})",
        )
    return None


def check_receipt(claim: Claim, policy: Policy) -> Violation | None:
    """Flag items over the receipt threshold with no receipt (POL-1.3)."""
    if claim.category == "Mileage":
        return None  # mileage is miles x rate; there is no receipt
    threshold = policy["submission"]["receipt_required_over"]
    if claim.amount > threshold and not claim.receipt_attached:
        return Violation(
            claim_id=claim.claim_id,
            code="missing_receipt",
            policy_ref="POL-1.3",
            detail=(f"${claim.amount:,.2f} over ${threshold:,.2f} no receipt"),
        )
    return None


SINGLE_CLAIM_RULES: list[Rule] = [check_late, check_receipt]


found = [
    v
    for claim in claims
    for rule in SINGLE_CLAIM_RULES
    if (v := rule(claim, policy)) is not None
]
print(pd.Series([v.code for v in found]).value_counts().to_string())

missing_receipt    14
late_submission    11


In [5]:
# 3b -boundary tests
base = next(c for c in claims if c.category == "Meals" and c.receipt_attached)


def days_before_submit(n: int) -> Claim:
    """Copy of base with the expense dated n days before submission."""
    return base.model_copy(
        update={"expense_date": base.submitted_date - timedelta(days=n)}
    )


# Late: 60 days is on  tim, 61 is late (POL-1.2: "more than 60")
assert check_late(days_before_submit(60), policy) is None
assert check_late(days_before_submit(61), policy).code == "late_submission"

# Receipt: exactly $75 is fine, $75.01 without a receipt is not
at_limit = base.model_copy(update={"amount": 75.00, "receipt_attached": False})
over = base.model_copy(update={"amount": 75.01, "receipt_attached": False})
assert check_receipt(at_limit, policy) is None
assert check_receipt(over, policy).code == "missing_receipt"


# Mileage never needs a receipt
mileage = base.model_copy(
    update={"category": "Mileage", "amount": 500.0, "receipt_attached": False}
)
assert check_receipt(mileage, policy) is None

# Anchor to know numbers: 01b found exactly 11 late claims
n_late = sum(v.code == "late_submission" for v in found)
assert n_late == 11, f"expected 11 late claims, got {n_late}"
print("boundary tests passed")

boundary tests passed


## 4: Cap Rules

**`check_cap`** handles all six flat per-category caps through one function by
delegating the arithmetic to `cap_usage` from the package (which returns the comparable
value and its cap, e.g. per-night for Lodging, per-person for Client Meals):

- Non-policy categories return `None` — the prohibited rule owns those.
- `cap_usage` returning `None` means the category is uncapped (or nights/attendees are
  missing), so there is nothing to compare.
- `round(value, 2)` before comparing avoids float-noise false positives
  (e.g. 7 nights × $249.99).
- `CAP_POLICY_REFS` maps each category to the right POL-2.x citation; `CAP_UNITS`
  makes details read naturally ("/night", "/person").

This cell also introduces **`apply_rules`** — the generic runner (every rule × every
claim, keep non-None results) reused for every rule group from here on.

Cell 4b tests the lodging city tiers ($350 high-cost vs $250 standard), float safety,
zero/missing attendees not crashing, uncapped categories never flagging, and that every
`over_cap` finding cites a POL-2.x section.

In [6]:
# 4 cap rules (POL-2.2, 2.3, 2.4, 2.7, 2.9, 2.10)

CAP_POLICY_REFS: dict[str, str] = {
    "Lodging": "POL-2.2",
    "Meals": "POL-2.3",
    "Client Meals": "POL-2.4",
    "Conference": "POL-2.7",
    "Home Office": "POL-2.9",
    "Office Supplies": "POL-2.10",
}
CAP_UNITS: dict[str, str] = {"Lodging": "/night", "Client Meals": "/person"}


def check_cap(claim: Claim, policy: Policy) -> Violation | None:
    """Flag amounts over their category cap (POL-2.x by category)."""
    if claim.category not in policy["categories"]:
        return None  # not a policy category; the prohibited rule owns it
    usage = cap_usage(claim.model_dump(), policy)
    if usage is None:
        return None  # uncapped category, or no nights/attendees
    value, cap = usage
    if round(value, 2) > cap:
        unit = CAP_UNITS.get(claim.category, "")
        return Violation(
            claim_id=claim.claim_id,
            code="over_cap",
            policy_ref=CAP_POLICY_REFS[claim.category],
            detail=(
                f"{claim.category} ${value:,.2f}{unit} "
                f"over cap ${cap:,.2f}{unit}"
            ),
        )
    return None


def apply_rules(
    claims: list[Claim], rules: list[Rule], policy: Policy
) -> list[Violation]:
    """Run every rule on every claim; keep all violations found."""
    return [
        violation
        for claim in claims
        for rule in rules
        if (violation := rule(claim, policy)) is not None
    ]


SINGLE_CLAIM_RULES = [check_late, check_receipt, check_cap]
found = apply_rules(claims, SINGLE_CLAIM_RULES, policy)

print(pd.Series([v.code for v in found]).value_counts().to_string())
print(f"claims flagged: {len({v.claim_id for v in found})}")

over_cap           18
missing_receipt    14
late_submission    11
claims flagged: 43


In [7]:
# 4b cap boundary tests
lodging = next(c for c in claims if c.category == "Lodging" and c.nights)


def lodging_at(city: str, nightly: float, nights: int = 2) -> Claim:
    """Copy of a lodging claim at a given nightly rate and city."""
    return lodging.model_copy(
        update={"city": city, "nights": nights, "amount": nightly * nights}
    )


# Tier caps: $350 high-cost, $250 standard (POL-2.2)
assert check_cap(lodging_at("New York, NY", 350.00), policy) is None
assert check_cap(lodging_at("New York, NY", 350.01), policy).code == "over_cap"
assert check_cap(lodging_at("Austin, TX", 250.00), policy) is None
assert check_cap(lodging_at("Austin, TX", 300.00), policy).policy_ref == (
    "POL-2.2"
)


# Float safety: 7 nights at $249.99 must not flag
assert check_cap(lodging_at("Austin, TX", 249.99, nights=7), policy) is None

# Client Meals: zero or missing attendees must not crash
client_meal = next(c for c in claims if c.category == "Client Meals")
for n in (0, None):
    no_guests = client_meal.model_copy(update={"attendees": n})
    assert check_cap(no_guests, policy) is None

# Uncapped categories never flag
airfare = next(c for c in claims if c.category == "Airfare")
assert (
    check_cap(airfare.model_copy(update={"amount": 9_999.0}), policy) is None
)

# Sanity: every over_cap flag cities a POL-2.x section
assert all(
    v.policy_ref.startswith("POL-2.") for v in found if v.code == "over_cap"
)
print("cap tests passed")

cap tests passed


## 5: Class, Vendor and Mileage Rules

Four conditional rules that depend on more than just an amount:

- **`check_fare_class` (POL-2.1):** premium airfare is allowed only as Business class
  on a flight of at least the policy's minimum hours *and* preapproved; anything else
  premium (First, short Business, unapproved) is a violation. Note the guard: a
  `None` fare class means "not applicable", while `None` flight hours on a Business
  fare fails the length requirement.
- **`check_rental_class` (POL-2.5):** rental car class must be on the allowed list.
- **`check_software_vendor` (POL-2.8):** software must come from an approved vendor.
  (Section 6 later *redefines* this name as a softer review flag — see that section.)
- **`check_mileage` (POL-2.6):** reimbursement may not exceed miles × the IRS rate
  *in force on the expense date*; `mileage_rate` walks dated rate bands from the policy
  and raises if a date falls outside every band rather than guessing.

Cell 5b sweeps the fare-class decision table, checks the rate change on Jul 1 2026,
and anchors against the generated set: exactly 6 premium-airfare violations and zero
false positives under POL-2.5/2.6/2.8.

In [8]:
# 5 fare/rental class, software vendor, mileage rate


def check_fare_class(claim: Claim, policy: Policy) -> Violation | None:
    """Flag premium airfare outside policy conditions (POL-2.1)."""
    if claim.category != "Airfare" or claim.fare_class is None:
        return None
    rules = policy["categories"]["Airfare"]
    if claim.fare_class in rules["allowed_fare_classes"]:
        return None
    min_hours = rules["business_class_min_flight_hours"]
    long_enough = (
        claim.flight_hours is not None and claim.flight_hours >= min_hours
    )
    if claim.fare_class == "Business" and long_enough and claim.preapproved:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="premium_class",
        policy_ref="POL-2.1",
        detail=(
            f"{claim.fare_class} on {claim.flight_hours}h flight, "
            f"preapproved={claim.preapproved} "
            f"(needs Business, >= {min_hours}h, preapproved)"
        ),
    )


def check_rental_class(claim: Claim, policy: Policy) -> Violation | None:
    """flag rental cars above the allowed classes (POL-2.5)."""
    if claim.category != "Ground Transport" or claim.rental_class is None:
        return None
    allowed = policy["categories"]["Ground Transport"][
        "allowed_rental_classes"
    ]
    if claim.rental_class in allowed:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="premium_class",
        policy_ref="POL-2.5",
        detail=f"rental class {claim.rental_class} not in {allowed}",
    )


def check_software_vendor(claim: Claim, policy: Policy) -> Violation | None:
    """Flag software bought from an unapproved vendor (POL-2.8)."""
    if claim.category != "Software":
        return None
    approved = policy["categories"]["Software"]["approved_vendors"]
    if claim.merchant in approved:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="prohibited_category",
        policy_ref="POL-2.8",
        detail=f"vendor {claim.merchant!r} not on approved list",
    )


def mileage_rate(expense_date: Date, policy: Policy) -> float:
    """Return the IRS mileage rate in force on a date (POL-2.6)."""
    for band in policy["categories"]["Mileage"]["rates"]:
        start = Date.fromisoformat(str(band["from"]))
        end = Date.fromisoformat(str(band["to"]))
        if start <= expense_date <= end:
            return band["rate"]
    raise ValueError(f"No mileage rate configured for {expense_date}")


def check_mileage(claim: Claim, policy: Policy) -> Violation | None:
    """Flag mileage claimed above miles x the dated rate (POL-2.6)."""
    if claim.category != "Mileage" or claim.miles is None:
        return None
    rate = mileage_rate(claim.expense_date, policy)
    allowed = round(claim.miles * rate, 2)
    if round(claim.amount, 2) > allowed:
        return Violation(
            claim_id=claim.claim_id,
            code="over_cap",
            policy_ref="POL-2.6",
            detail=(
                f"${claim.amount:,.2f} > {claim.miles} mi x ${rate}/mi "
                f"= ${allowed:,.2f}"
            ),
        )
    return None


SINGLE_CLAIM_RULES = [
    check_late,
    check_receipt,
    check_cap,
    check_fare_class,
    check_rental_class,
    check_mileage,
    check_software_vendor,
]
found = apply_rules(claims, SINGLE_CLAIM_RULES, policy)

print(pd.Series([v.code for v in found]).value_counts().to_string())
print(f"claims flagged: {len({v.claim_id for v in found})}")

over_cap           18
missing_receipt    14
late_submission    11
premium_class       6
claims flagged: 49


In [9]:
# 5b -class, vendor and mileage tests
flight = next(c for c in claims if c.category == "Airfare")


def fly(fare: str, hours: float | None, pre: bool) -> Claim:
    """Copy of an airfare claim with given fare, hours, preapproval."""
    return flight.model_copy(
        update={"fare_class": fare, "flight_hours": hours, "preapproved": pre}
    )


assert check_fare_class(fly("Economy", 2.0, False), policy) is None
assert check_fare_class(fly("Premium Economy", 9.0, False), policy) is None
assert check_fare_class(fly("Business", 6.0, True), policy) is None  # boundary
assert check_fare_class(fly("Business", 5.9, True), policy) is not None
assert check_fare_class(fly("Business", 8.0, False), policy) is not None
assert check_fare_class(fly("Business", None, True), policy) is not None
assert check_fare_class(fly("First", 12.0, True), policy) is not None

# Mileage rate changes on Jul 1, 2026
assert mileage_rate(Date(2026, 6, 30), policy) == 0.725
assert mileage_rate(Date(2026, 7, 1), policy) == 0.76

# Data-anchored expectations (generated set)
codes = pd.Series([v.code for v in found])
refs = pd.Series([v.policy_ref for v in found])
assert (refs == "POL-2.1").sum() == 6, "expected 6 premium airfare"
for ref in ("POL-2.5", "POL-2.6", "POL-2.8"):
    assert (refs == ref).sum() == 0, f"false positives under {ref}"
print("class, vendor and mileage tests passed")

class, vendor and mileage tests passed


## 6: Flags (prohibited hints, unapproved vendor)

Introduces the **`Flag`** model: same shape as `Violation`, but a different meaning —
a reason for closer review (by a human or the agent), *not* a denial.

- **`check_prohibited_hint` (POL-3.x):** `compile_prohibited` builds one
  case-insensitive, whole-word regex per prohibited item (with an optional plural
  `s`), matched against merchant + description text. Keyword matching on free text is
  inherently fuzzy, which is exactly why this is a *hint* flag rather than a violation.
  Exception: alcohol keywords are skipped for Client Meals, where POL-2.4 allows it.
- **`check_software_vendor` (POL-2.8), redefined:** this *shadows* the section-5
  function of the same name and returns a `Flag` instead of a `Violation` — an
  unapproved vendor now routes to review ("needs IT approval") rather than denial.
  The earlier `SINGLE_CLAIM_RULES` list still holds the original denying version it
  captured in section 5, so both behaviors coexist deliberately.

Cell 6b exercises word-boundary behavior ("Corner Bar" hits, "Delta … Airline" and
"Workspace Lunch" don't despite containing bar/space-adjacent substrings), the client-
meal alcohol exception, and prints one *documented, intentional* false positive
("Fine dining" at a client meal) — acceptable because a hint only triggers review.

In [10]:
# 6a - Flags: prohibited-keyword hints, unapproved vendors

FlagCode = Literal[
    "prohibited_hint",
    "unapproved_vendor",
    "high_value",
    "near_cap",
]


class Flag(BaseModel):
    """A reason for closer review (human or agent), not a denial."""

    model_config = ConfigDict(frozen=True, extra="forbid")

    claim_id: str
    code: FlagCode
    policy_ref: str = Field(pattern=r"^POL-\d+\.\d+$")
    detail: str = Field(min_length=1)


def compile_prohibited(policy: Policy) -> list[tuple[dict, re.Pattern[str]]]:
    """Build one case-insensitive, whole-word regex per POL-3 item."""
    compiled = []
    for item in policy["prohibited"]:
        words = "|".join(re.escape(word) for word in item["keywords"])
        pattern = re.compile(rf"\b(?:{words})s?\b", re.IGNORECASE)
        compiled.append((item, pattern))
    return compiled


PROHIBITED_PATTERNS = compile_prohibited(policy)


def check_prohibited_hint(claim: Claim, policy: Policy) -> Flag | None:
    """Hint at a possible POL-3 item from merchant/description text."""
    text = f"{claim.merchant} {claim.description}"
    for item, pattern in PROHIBITED_PATTERNS:
        alcohol = item["code"] == "ALCOHOL_NO_CLIENT"
        if alcohol and claim.category == "Client Meals":
            continue  # POL-2.4: alcohol is allowed at a client meal
        if match := pattern.search(text):
            return Flag(
                claim_id=claim.claim_id,
                code="prohibited_hint",
                policy_ref=item["policy_ref"],
                detail=f"{item['code']}: matched {match.group(0)!r}",
            )
    return None


def check_software_vendor(claim: Claim, policy: Policy) -> Flag | None:
    """Flag software from a vendor that needs IT approval (POL-2.8)."""
    if claim.category != "Software":
        return None
    approved = policy["categories"]["Software"]["approved_vendors"]
    if claim.merchant in approved:
        return None
    return Flag(
        claim_id=claim.claim_id,
        code="unapproved_vendor",
        policy_ref="POL-2.8",
        detail=f"vendor {claim.merchant!r} needs IT approval",
    )


FLAG_RULES = [check_prohibited_hint, check_software_vendor]
flags = apply_rules(claims, FLAG_RULES, policy)

hints = pd.DataFrame(
    [
        {
            "claim_id": f.claim_id,
            "category": c.category,
            "text": f"{c.merchant} | {c.description}",
            "detail": f.detail,
        }
        for f in flags
        for c in claims
        if c.claim_id == f.claim_id
    ]
)
print(pd.Series([f.code for f in flags]).value_counts().to_string())
hints

prohibited_hint    26


,claim_id,category,text,detail
0,R-E001-01-L3,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
1,R-E001-08-L4,Lodging,Marriott | In-room movies,WELLNESS_ENTERTAINMENT: matched 'In-room'
2,R-E004-06-L4,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
3,R-E005-02-L4,Lodging,Hotel Spa | Spa massage during trip,WELLNESS_ENTERTAINMENT: matched 'Spa'
4,R-E005-04-L2,Meals,Pinstripe Bar & Lounge | Drinks with coworkers...,ALCOHOL_NO_CLIENT: matched 'Bar'
5,R-E006-01-L1,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
6,R-E006-02-L2,Home Office,Best Buy | AirPods Pro for personal commute,PERSONAL_ITEM: matched 'personal'
7,R-E006-03-L3,Ground Transport,City Parking Enforcement | Parking ticket near...,FINE: matched 'Parking ticket'
8,R-E006-03-L4,Airfare,Delta Air Lines | Airfare for spouse to join trip,FAMILY_TRAVEL: matched 'spouse'
9,R-E006-10-L2,Ground Transport,City Parking Enforcement | Parking ticket near...,FINE: matched 'Parking ticket'


In [11]:
# 6b hint tests
meal = next(c for c in claims if c.category == "Meals")
client = next(c for c in claims if c.category == "Client Meals")


def says(base: Claim, merchant: str, description: str) -> Claim:
    """Copy of a claim with new merchant and description text."""
    return base.model_copy(
        update={"merchant": merchant, "description": description}
    )


hit = check_prohibited_hint(says(meal, "Corner Bar", "Team drinks"), policy)
assert hit is not None and hit.policy_ref == "POL-3.1"
assert (
    check_prohibited_hint(says(client, "Corner Bar", "Wine w/ Acme"), policy)
    is None
)
assert (
    check_prohibited_hint(says(meal, "Delta", "Airline ticket"), policy)
    is None
)
assert (
    check_prohibited_hint(says(meal, "Hotel", "In-room movies"), policy)
    is not None
)
assert (
    check_prohibited_hint(says(meal, "Cafe", "Workspace Lunch"), policy)
    is None
)


# Known false positive, documented on purpose: a hint, not a denial
fp = check_prohibited_hint(says(client, "Le Bernardin", "Fine dining"), policy)
print(f"known false positive: {fp.detail if fp else 'none'}")
print("hint tests passed")

known false positive: FINE: matched 'Fine'
hint tests passed


## 7: Escalation Flags

Two more review flags based on money, not wrongdoing:

- **`check_high_value` (POL-5.1):** any single item *strictly over* the high-value
  threshold goes to review, even if otherwise compliant.
- **`check_near_cap` (POL-5.2):** amounts landing inside the band
  `[cap × (1 − pct), cap]` get flagged as suspiciously close to the limit. Amounts
  *over* the cap are excluded on purpose — `check_cap` owns those as violations —
  which makes the two rules a clean handoff with no gap and no overlap.

Cell 7b pins all four edges of the Office Supplies band ($134.99 / $135.00 / $150.00 /
$150.01) and verifies the handoff: $150.01 is no longer `near_cap` but *is* `over_cap`.

In [12]:
# 7a Escalation flags: high value (POL-5.1), near cap (POL-5.2)


def check_high_value(claim: Claim, policy: Policy) -> Flag | None:
    """Flag a single item over the high-value threshold (POL-5..1)."""
    threshold = policy["escalation"]["high_value_threshold"]
    if claim.amount <= threshold:
        return None
    return Flag(
        claim_id=claim.claim_id,
        code="high_value",
        policy_ref="POL-5.1",
        detail=f"${claim.amount:,.2f} over ${threshold:,.2f}",
    )


def check_near_cap(claim: Claim, policy: Policy) -> Flag | None:
    """Flag amounts within the near-cap band below a cap (POL-5.2)."""
    if claim.category not in policy["categories"]:
        return None  # same guard as check_cap: cap_usage raises KeyError
    usage = cap_usage(claim.model_dump(), policy)
    if usage is None:
        return None  # uncapped category, or no nights/attendees
    value, cap = usage
    value = round(value, 2)
    pct = policy["escalation"]["near_cap_pct"]
    lower = round(cap * (1 - pct), 2)
    if not lower <= value <= cap:
        return None  # well under the band, or over cap (check_cap owns it)
    unit = CAP_UNITS.get(claim.category, "")
    return Flag(
        claim_id=claim.claim_id,
        code="near_cap",
        policy_ref="POL-5.2",
        detail=(
            f"{claim.category} ${value:,.2f}{unit} is "
            f"{value / cap:.0%} of cap ${cap:,.2f}{unit}"
        ),
    )


ESCALATION_RULES = [check_high_value, check_near_cap]
escalations = apply_rules(claims, ESCALATION_RULES, policy)

print(pd.Series([f.code for f in escalations]).value_counts().to_string())
print(f"claims escalated: {len({f.claim_id for f in escalations})}")

high_value    15
near_cap       4
claims escalated: 19


In [13]:
# 7b - Boundary tests for escalation flags

base = next(c for c in claims if c.category == "Office Supplies")


def with_amount(claim: Claim, amount: float) -> Claim:
    """Return a copy with a new amount (tests only; skips validation)."""
    return claim.model_copy(update={"amount": amount})


# POL-5.1: strictly "over" $1,000
assert check_high_value(with_amount(base, 1000.00), policy) is None
flag = check_high_value(with_amount(base, 1000.01), policy)
assert flag is not None and flag.code == "high_value"

# POL-5.2: band is [135.00, 150.00] for Office Supplies
assert check_near_cap(with_amount(base, 134.99), policy) is None
assert check_near_cap(with_amount(base, 135.00), policy).code == "near_cap"
assert check_near_cap(with_amount(base, 150.00), policy).code == "near_cap"
assert check_near_cap(with_amount(base, 150.01), policy) is None
assert check_cap(with_amount(base, 150.01), policy) is not None  # handoff

print("7b: all escalation boundary tests passed")

7b: all escalation boundary tests passed


## 8: Duplicates & Splits (cross-claim rules)

These rules need the whole claim list, so they use their own signature instead of
`Rule`. `claims_frame` is the shared prep: dump claims to a DataFrame, convert amounts
to integer cents (exact equality — no float comparisons), and sort by submission date
with original file order breaking same-day ties, so "first" is well defined.

- **`check_duplicates` (POL-1.4):** groups on employee + expense date + merchant +
  amount; the earliest submission in each group stands, every repeat is denied, and
  the detail names the original claim it duplicates.
- **`check_splits` (POL-1.5):** groups on employee + expense date + merchant +
  *category* (amounts differ by construction in a split). A group is a split only if
  the parts *total* over the purchase cap while **each part individually fits under
  it** — i.e. the cap was dodged by splitting. If any single part already exceeds the
  cap, `check_cap` owns it. All parts of a split are denied, each detail naming its
  partners. Claims already denied as duplicates are skipped (`skip_ids`) so one claim
  isn't condemned twice for overlapping reasons.

Cell 8c proves the *later* submission is always the duplicate regardless of list
order, that two under-cap parts totalling over the cap both get denied, and that a
total within the cap is not a split.

In [14]:
# 8a - Cross-claim rule: duplicates (POL-1.4)

DUPLICATE_KEYS = ["employee_id", "expense_date", "merchant", "amount_cents"]


def claims_frame(claims: list[Claim]) -> pd.DataFrame:
    """Return claims as a DataFrame,  earliest submission first."""
    df = pd.DataFrame([claim.model_dump() for claim in claims])
    df["position"] = range(len(df)) # file order breaks same-day ties
    df["amount_cents"] = (df["amount"] * 100).round().astype(int)
    return df.sort_values(["submitted_date", "position"], kind="stable")

def check_duplicates(claims: list[Claim]) -> list[Violation]:
    """Deny repeats of an earlier claim; the first one stands (POL-1.4)."""
    df = claims_frame(claims)
    df["original_id"] = df.groupby(DUPLICATE_KEYS)["claim_id"].transform("first")
    repeats = df[df.duplicated(DUPLICATE_KEYS, keep="first")]
    return [
        Violation(
            claim_id=row.claim_id,
            code="duplicate",
            policy_ref="POL-1.4",
            detail=f"same employee/date/merchant/amount as {row.original_id}",
        )
        for row in repeats.itertuples()
    ]


In [15]:
# 8b - Cross-claim rule: split purchases (POL-1.5)

SPLIT_KEYS = ["employee_id", "expense_date", "merchant", "category"]
PURCHASE_CAP_KEYS = ("per_purchase_cap", "per_event_cap")



def purchase_caps(policy: Policy) -> dict[str, float]:
    """Map each category that has a single-purchase cap to the cap."""
    return {
        category: rules[key]
        for category, rules in policy["categories"].items()
        for key in PURCHASE_CAP_KEYS
        if key in rules
        }


def check_splits(
        claims: list[Claim], policy: Policy, skip_ids: set[str]
) -> list[Violation]:
    """Deny every part of a purchase split to stay under a cap (POL-1.5)."""
    caps = purchase_caps(policy)
    df = claims_frame(claims)
    df = df[df["category"].isin(caps) & ~df["claim_id"].isin(skip_ids)]
    found = []
    for _, group in df.groupby(SPLIT_KEYS):
        if len(group) < 2:
            continue
        cap = caps[group["category"].iloc[0]]
        total = group["amount"].sum()
        if total <= cap or (group["amount"] > cap).any():
            continue # no dodge: fits the cap, or a part is over_cap
        part_ids = list(group["claim_id"]
        )
        found.extend(
            Violation(
                claim_id=claim_id,
                code="split_transaction",
                policy_ref="POL-1.5",
                detail=(
                    f"split with "
                    f"{', '.join(p for p in part_ids if p != claim_id)}: "
                    f"total ${total:,.2f} over cap ${cap:,.2f}"
                ),
            )
            for claim_id in part_ids
        )
    return found


duplicates = check_duplicates(claims)
duplicate_ids = {v.claim_id for v in duplicates}
splits = check_splits(claims, policy, skip_ids=duplicate_ids)

print(f"duplicates: {len(duplicates)} splits: {len(splits)}")
for v in duplicates + splits:
    print(f"{v.claim_id:<14} {v.code:<18} {v.detail}")

already_found = {v.claim_id for v in found} # single-claim rules, cells 3-5
cross_ids = {v.claim_id for v in duplicates + splits}
print(f"overlap with single-claim violations: {len(already_found & cross_ids)}")

duplicates: 7 splits: 6
R-E003-06-L3   duplicate          same employee/date/merchant/amount as R-E003-06-L1
R-E005-06-L2   duplicate          same employee/date/merchant/amount as R-E005-06-L1
R-E004-01-L4   duplicate          same employee/date/merchant/amount as R-E004-01-L2
R-E005-07-L4   duplicate          same employee/date/merchant/amount as R-E005-07-L1
R-E005-03-L3   duplicate          same employee/date/merchant/amount as R-E005-03-L2
R-E003-08-L3   duplicate          same employee/date/merchant/amount as R-E003-08-L2
R-E010-06-L3   duplicate          same employee/date/merchant/amount as R-E010-06-L2
R-E002-01-L1   split_transaction  split with R-E002-01-L2: total $661.06 over cap $500.00
R-E002-01-L2   split_transaction  split with R-E002-01-L1: total $661.06 over cap $500.00
R-E003-03-L2   split_transaction  split with R-E003-03-L4: total $625.50 over cap $500.00
R-E003-03-L4   split_transaction  split with R-E003-03-L2: total $625.50 over cap $500.00
R-E004-06-L1   split_

In [16]:
# 8c - Tests for duplicate and split rules

base = next(c for c in claims if c.category == "Home Office")
later = base.submitted_date + timedelta(days=1)

# POL- 1.4: the LATER submission is the duplicate, whatever the list order
copy = base.model_copy(update={"claim_id": "T-DUP", "submitted_date": later})
dups = check_duplicates([copy, base])
assert [v.claim_id for v in dups] == ["T-DUP"]
assert dups[0].detail.endswith(base.claim_id)

# POL-1.5: two parts under the $500 cap, total over -> both parts denied
part1 = base.model_copy(update={"claim_id": "T-P1", "amount": 300.00})
part2 = part1.model_copy(update={"claim_id": "T-P2", "amount": 250.00, "submitted_date": later})
splits_found = check_splits([part1, part2], policy, skip_ids=set())
assert sorted(v.claim_id for v in splits_found) == ["T-P1", "T-P2"]

# Total within the cap -> not a split
small = part2.model_copy(update={"amount": 150.00})
assert check_splits([part1, small], policy, skip_ids=set()) == []


print("8c: duplicate and split tests passed")

8c: duplicate and split tests passed


## 9: run_results()

Assembles everything into one callable pipeline:

- **`collect_findings`** runs all rule groups in order — single-claim rules, then
  duplicates, then splits (skipping duplicate IDs), then flag + escalation rules —
  returning violations and flags separately.
- **`rule_decision`** encodes the precedence from POL-4/POL-5: any violation ⇒
  **deny**; else any flag ⇒ **review**; else **approve**.
- **`run_rules`** groups findings by claim and emits exactly one row per claim with
  the decision, violation codes, flag codes, cited policy sections, and human-readable
  details — the full audit trail for each claim.

Invariant checks: one row per claim, unique IDs, and every denied row carries at least
one violation (a deny can never be unexplained).

In [17]:
# 9 - run_results(): all rules, one row per claim (POL-4, POL-5)

Decision = Literal["approve", "deny", "review"]


def collect_findings(claims: list[Claim], policy: Policy) -> tuple[list[Violation], list[Flag]]:
    """Run every rule group; return violations and flags separately."""
    violations = apply_rules(claims, SINGLE_CLAIM_RULES, policy)
    duplicates = check_duplicates(claims)
    duplicate_ids = {v.claim_id for v in duplicates}
    violations += duplicates
    violations += check_splits(claims, policy, skip_ids=duplicate_ids)
    flags = apply_rules(claims, FLAG_RULES + ESCALATION_RULES, policy)
    return violations, flags


def rule_decision(violation_codes: list[str], flag_codes: list[str]) -> Decision:
    """Deny beats review beats approve (POL-4, POL-5)."""
    if violation_codes:
        return "deny"
    if flag_codes:
        return "review"
    return "approve"


def run_rules(claims: list[Claim], policy: Policy) -> pd.DataFrame:
    """Return one row per claim: rule decision plus every finding."""
    violations, flags = collect_findings(claims, policy)
    by_claim: dict[str, list[Violation | Flag]] = defaultdict(list)
    for finding in [*violations, *flags]:
        by_claim[finding.claim_id].append(finding)

    rows = []
    for claim in claims:
        claim_findings = by_claim[claim.claim_id]
        violation_codes = [
            f.code for f in claim_findings if isinstance(f, Violation)
        ]
        flag_codes = [f.code for f in claim_findings if isinstance(f, Flag)]
        rows.append(
            {
                "claim_id": claim.claim_id,
                "rule_decision": rule_decision(violation_codes, flag_codes),
                "violations": violation_codes,
                "flags": flag_codes,
                "policy_refs":sorted({f.policy_ref for f in claim_findings}),
                "details": [f.detail for f in claim_findings],
            }
        )
    return pd.DataFrame(rows)

rule_results = run_rules(claims, policy)

assert len(rule_results) == len(claims), "one row per claim"
assert rule_results["claim_id"].is_unique
denied = rule_results["rule_decision"] == "deny"
assert rule_results.loc[denied, "violations"].map(len).gt(0).all()


print(rule_results["rule_decision"].value_counts().to_string())
print(rule_results[denied].head(5).to_string())

rule_decision
approve    237
deny        62
review      31
        claim_id rule_decision           violations         flags         policy_refs                                                                                                     details
9   R-E001-05-L2          deny           [over_cap]  [high_value]  [POL-2.9, POL-5.1]                                          [Home Office $1,196.21 over cap $500.00, $1,196.21 over $1,000.00]
16  R-E001-07-L3          deny      [premium_class]  [high_value]  [POL-2.1, POL-5.1]  [Business on 3.9h flight, preapproved=True (needs Business, >= 6h, preapproved), $1,927.76 over $1,000.00]
25  R-E001-10-L2          deny    [missing_receipt]            []           [POL-1.3]                                                                            [$232.09 over $75.00 no receipt]
26  R-E002-01-L1          deny  [split_transaction]            []           [POL-1.5]                                                   [split with R-E002-01-L2: tot

## 10: Score vs Labels

The only place ground-truth labels appear — strictly for **evaluation**; nothing here
feeds back into a rule.

Setup: labels and amounts are merged onto `rule_results` with `validate="one_to_one"`
guards. One scoring adjustment is needed first: labels mark only one half of each
split purchase as `split_transaction`, but POL-1.5 correctly denies *both* halves.
The `same_purchase` mask finds in-policy claims whose **only** violation is
`split_transaction` and whose detail names a labeled split partner (via the
`CLAIM_ID` regex) — those are the same purchase, not false positives.

Four read-outs follow:

1. **Catch rate & false positives** — how many out-of-policy claims the rules deny,
   whether the labeled violation type appears among the rule's codes, and how many
   in-policy claims were wrongly denied (after the split adjustment).
2. **Confusion matrix** — labeled decision (rows) vs rule decision (columns).
3. **Rules-only baseline for notebook 06** — false-approval rate, dollars leaked
   through false approvals, and label-deny claims parked in review (dollars at risk).
4. **What rules miss** — out-of-policy claims the rules did *not* deny, broken down
   by violation type: exactly the gap the LLM agent must close in notebook 05.

Anchors lock in the headline result: 59 out-of-policy claims caught, 3 split halves
correctly denied with their purchase, and **zero** false positives.

In [18]:
# 10 - Score rules vs labels (evaluation only: labels never feed a rule)
CLAIM_ID = re.compile(r"R-E\d{3}-\d{2}-L\d+")


def named_claims(details: list[str]) -> set[str]:
    """Return every claim ID mentioned in a row's finding details."""
    return {cid for detail in details for cid in CLAIM_ID.findall(detail)}


labels = load_labels(GENERATED_DIR)

assert set(labels["claim_id"]) == set(rule_results["claim_id"]), "ID mismatch"
assert labels["in_policy"].dtype == bool, "in_policy must load as bool"

scored = rule_results.merge(
    labels, on="claim_id", validate="one_to_one"
).merge(
    claims_df[["claim_id", "amount"]], on="claim_id", validate="one_to_one"
)

out_of_policy = ~scored["in_policy"]
rule_deny = scored["rule_decision"] == "deny"
label_deny = scored["expected_decision"] == "deny"

# Splits are scored per purchase (POL-1.5): a clean half denied only
# because its labeled partner is a split is the same purchase, not a FP.
labeled_splits = set(
    scored.loc[scored["violation_type"] == "split_transaction", "claim_id"]
)
split_only = scored["violations"].map(
    lambda codes: codes == ["split_transaction"]
)
same_purchase = (
    scored["in_policy"]
    & split_only
    & scored["details"].map(lambda ds: bool(named_claims(ds) & labeled_splits))
)

# 1. Catch rate and false positives
caught = out_of_policy & rule_deny
false_pos = scored["in_policy"] & rule_deny & ~same_purchase
type_match = pd.Series(
    [
        label in codes
        for label, codes in zip(scored["violation_type"], scored["violations"])
    ],
    index=scored.index,
)
print(f"caught: {caught.sum()} of {out_of_policy.sum()} out-of-policy")
print(f"  labeled type among rule codes: {(caught & type_match).sum()}")
print(f"split halves denied with their purchase: {same_purchase.sum()}")
print(f"false positives (in policy, denied): {false_pos.sum()}")

# 2. Decision confusion matrix: rows = label, columns = rules
print()
print(
    pd.crosstab(
        scored["expected_decision"],
        scored["rule_decision"],
        margins=True,
    )
)

# 3. Rules-only baseline for notebook 06
false_approve = label_deny & (scored["rule_decision"] == "approve")
to_review = label_deny & (scored["rule_decision"] == "review")
fa_rate = false_approve.sum() / label_deny.sum()
leaked = scored.loc[false_approve, "amount"].sum()
at_risk = scored.loc[to_review, "amount"].sum()
print()
print(
    f"false-approval rate: {fa_rate:.1%} "
    f"({false_approve.sum()} of {label_deny.sum()})"
)
print(f"dollars leaked: ${leaked:,.2f}")
print(f"label deny sent to review: {to_review.sum()} (${at_risk:,.2f})")

# 4. What rules miss: the agent's job in notebook 05
missed = out_of_policy & ~rule_deny
print()
print(
    pd.crosstab(
        scored.loc[missed, "violation_type"],
        scored.loc[missed, "rule_decision"],
        margins=True,
    )
)

# Anchors: B + A applied (Session 8)
assert caught.sum() == 59, f"expected 59 caught, got {caught.sum()}"
assert same_purchase.sum() == 3, "one clean half per split purchase"
assert false_pos.sum() == 0, "rules denied an in-policy claim"

caught: 59 of 82 out-of-policy
  labeled type among rule codes: 57
split halves denied with their purchase: 3
false positives (in policy, denied): 0

rule_decision      approve  deny  review  All
expected_decision                            
approve                222     3       6  231
deny                     6    59      17   82
review                   9     0       8   17
All                    237    62      31  330

false-approval rate: 7.3% (6 of 82)
dollars leaked: $2,394.30
label deny sent to review: 17 ($2,900.89)

rule_decision        approve  review  All
violation_type                           
personal_extension         6       0    6
prohibited_category        0      17   17
All                        6      17   23


---

## Summary

**What was built:** a fully deterministic, policy-cited rules engine — typed
`Claim`/`Violation`/`Flag` models, seven single-claim rules, two review-flag rules, two
escalation flags, two cross-claim rules (duplicates, splits), and a `run_rules()`
pipeline producing one auditable decision row per claim.

**Headline results (asserted, not just printed):**

- **59** out-of-policy claims caught by rules, with **0 false positives** — rules never
  deny an in-policy claim.
- **3** clean split halves correctly denied together with their labeled partner
  (splits are judged per purchase, per POL-1.5).
- Every denial cites the exact policy section it enforces.

**What rules can and cannot do:**

- Rules excel at *mechanical* policy: thresholds, caps, dates, exact-match duplicates,
  arithmetic splits. They are cheap, instant, explainable, and perfectly repeatable.
- Rules cannot judge *semantics*: whether a description masks a prohibited purchase,
  whether a business purpose is plausible, or whether a vendor name on review really
  needs IT approval. Those cases surface in the section-10 "missed" table and in the
  review queue — this is precisely the work handed to the LLM agent in **notebook 05**.
- The section-10 metrics (false-approval rate, dollars leaked, dollars at risk in
  review) form the **rules-only baseline** that **notebook 06** uses to measure whether
  the agent actually adds value over plain rules.

**Patterns worth reusing:** config-driven thresholds with a code↔config drift check,
frozen Pydantic models as rule contracts, violation-vs-flag severity split with
deny > review > approve precedence, boundary tests at every threshold edge, and anchor
asserts that pin known dataset counts so regressions fail loudly.